# 🔧 Wrench — a grounded bike-repair copilot

**Point at a bike → get every part labeled on *your own photo*, then turn a plain-English symptom into a step-by-step repair plan as guaranteed-parseable JSON.**

Built on [Perceptron Mk1](https://docs.perceptron.inc). Any model can *describe* a bike — the interesting part is **grounding** (a tight box around *the rear derailleur in this exact image*) and **structured output** (a diagnosis that always validates against a schema).

This notebook is **self-contained** — run the cells top to bottom. You'll need a free API key from [platform.perceptron.inc](https://platform.perceptron.inc).

## 1. Setup

In [ ]:
!pip install -q perceptron pillow pydantic

In [ ]:
import getpass
from perceptron import configure

# Paste your key when prompted (input is hidden). Get one at platform.perceptron.inc
API_KEY = getpass.getpass('Perceptron API key: ')

MODEL = 'perceptron-mk1'  # flagship image+video model with reasoning
configure(provider='perceptron', model=MODEL, api_key=API_KEY)
print('Configured', MODEL)

## 2. Choose a bike photo

Upload your own for the full effect (the whole point is labeling *your* bike). Or set `IMAGE` to a local path / public URL and skip the upload.

In [ ]:
IMAGE = ''  # <- optionally put a local path or public image URL here

if not IMAGE:
    from google.colab import files  # Colab-only helper
    uploaded = files.upload()       # opens a file picker
    IMAGE = next(iter(uploaded))    # use the first uploaded file

print('Using image:', IMAGE)

## 3. Anatomy mode — label the parts on *your* bike

`detect()` returns grounded boxes on a normalized 0–1000 grid; `boxes_to_pixels()` maps them to the image size. The class list is an **open vocabulary** — just strings, no training. Edit it to support any bike style.

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from PIL import Image, ImageDraw, ImageFont
from perceptron import detect, image

PARTS = [
    'saddle', 'seatpost', 'handlebar', 'stem', 'brake lever',
    'front wheel', 'rear wheel', 'tire', 'chain', 'chainring',
    'crank arm', 'pedal', 'front derailleur', 'rear derailleur',
    'cassette', 'disc brake rotor',
]

def load(src):
    """Load a local path or URL into a PIL image."""
    if src.startswith(('http://', 'https://')):
        with urlopen(src) as r:
            return Image.open(BytesIO(r.read())).convert('RGB')
    return Image.open(src).convert('RGB')

PALETTE = ['#e6194B','#3cb44b','#4363d8','#f58231','#911eb4','#42d4f4',
           '#f032e6','#bfef45','#469990','#9A6324','#800000','#000075']

def font(sz=18):
    try:
        return ImageFont.truetype('DejaVuSans-Bold.ttf', sz)
    except OSError:
        return ImageFont.load_default()

def draw_boxes(img, boxes):
    out = img.convert('RGB').copy(); d = ImageDraw.Draw(out); f = font()
    for i, b in enumerate(boxes):
        c = PALETTE[i % len(PALETTE)]
        x1,y1,x2,y2 = int(b.top_left.x),int(b.top_left.y),int(b.bottom_right.x),int(b.bottom_right.y)
        label = b.mention or 'part'
        d.rectangle([x1,y1,x2,y2], outline=c, width=3)
        tb = d.textbbox((0,0), label, font=f); tw,th = tb[2]-tb[0], tb[3]-tb[1]
        ly = y1-th-6 if y1-th-6 > 0 else y1+4
        d.rectangle([x1,ly,x1+tw+8,ly+th+6], fill=c)
        d.text((x1+4,ly+3), label, fill='white', font=f)
    return out

In [ ]:
# One call does the detection.
result = detect(image(IMAGE), classes=PARTS)

pil = load(IMAGE)
boxes = result.boxes_to_pixels(width=pil.width, height=pil.height) or []
print(f'Detected {len(boxes)} parts:')
for b in boxes:
    print('  -', b.mention)

annotated = draw_boxes(pil, boxes)
annotated  # renders inline in Colab

## 4. Diagnose mode — symptom → structured repair plan

We hand the model a **Pydantic schema** and get back JSON that's *guaranteed* to validate into it (constrained decoding). That's what makes the output safe to feed a UI, a work order, or a parts-ordering system.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from perceptron import perceive, pydantic_format, text

class RepairStep(BaseModel):
    order: int
    instruction: str
    part: Optional[str] = None
    caution: Optional[str] = None

class Diagnosis(BaseModel):
    symptom: str
    likely_cause: str
    affected_parts: list[str]
    severity: Literal['low','medium','high']
    difficulty: Literal['beginner','intermediate','advanced']
    tools_needed: list[str]
    estimated_time_min: int
    steps: list[RepairStep]

SYMPTOM = 'gears skip when I pedal hard'  # <- describe your problem

@perceive(response_format=pydantic_format(Diagnosis), model=MODEL, reasoning=True)
def diagnose(src, complaint):
    return image(src) + text(
        f"You are a bike mechanic. A rider reports: '{complaint}'. Inspect the bike "
        "and return a structured repair diagnosis. Prefer the simplest likely cause. "
        "Keep steps ordered and beginner-friendly; only list tools that are needed."
    )

dx = Diagnosis.model_validate_json(diagnose(IMAGE, SYMPTOM).text)  # always parses
print(f'Likely cause : {dx.likely_cause}')
print(f'Severity     : {dx.severity}   Difficulty: {dx.difficulty}')
print(f'Tools        : {", ".join(dx.tools_needed)}')
print(f'Est. time    : {dx.estimated_time_min} min\n')
for s in dx.steps:
    line = f'{s.order}. {s.instruction}'
    if s.caution:
        line += f'  ⚠️ {s.caution}'
    print(line)

## 5. (Optional) Focus — close-inspect a fine detail

`focus=True` lets the model zoom into a region and reason on the crop — good for chain wear, a bent hanger, cable fray. Point it at a close-up for best results.

In [ ]:
@perceive(focus=True, model=MODEL, reasoning=True)
def inspect_detail(src, question):
    return image(src) + text(question)

answer = inspect_detail(IMAGE, 'Look closely: is the chain worn or stretched, and is the rear derailleur aligned?')
print(answer.text)

---

**That's Wrench.** Grounded labels on a real photo + a schema-guaranteed diagnosis, in ~40 lines of model calls. Modular library + CLI version: see the repo. Next natural mode: swap `image()` → `video()` with `expects="clip"` to find *when* a problem shows up under load.